# Caso J · 03 Series temporales en InfluxDB para tráfico

> _Tutorial · Caso de uso: **J — Tráfico + YOLO** · Capa Medallion: **bronce → plata** · Spec: `docs/specs/synthetic-bms/01-product-spec.md`_

Material docente del proyecto **CAPTIA Synthetic Data BMS** — IES Dr. Lluís Simarro,
Curso de Especialización IA & Big Data 2025-2026.


## 1. Objetivo

Tomar el mock `traffic_camera_mock.csv` y construir line protocol con los tags `domain_id=traffic_cameras`, `site_id=valencia`, `asset_id=DGT_CAM_*`.


## 2. Qué se aprende

- Mapping camera_id → asset_id.
- 3 variables: count / level / confidence.
- Por qué los conteos van como counter o analog_gauge.


## 3. Contexto del caso de uso

Plata para Caso J.


## 4. Relación con CENTINELA+

Independiente del aula.


## 5. Relación con Medallion

Bronce → plata.


## 6. Datos de entrada

Mock 7 días × 15 min × 2 cámaras.


## 7. Schema CAPTIA esperado

`vehicle_count` (analog_gauge), `congestion_level` (analog_gauge), `detection_confidence` (analog_gauge).


## 8. Setup y variables de entorno

Cargamos las variables de entorno (`.env`), inicializamos `numpy` con `seed=42` y aplicamos el estilo de plotting compartido. Los helpers viven en `notebooks/_common/`.


In [ ]:
# Setup canónico — todos los notebooks didácticos lo usan
from __future__ import annotations

import sys
from pathlib import Path

ROOT = Path.cwd()
while ROOT.name and not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from notebooks._common.captia_schema import (
    CANONICAL_TAGS, MEASUREMENT_TELEMETRY, MEASUREMENT_FAULT_LABELS,
    DEFAULT_BUCKET_RETENTIONS, KNOWN_VARIABLES,
    build_topic, build_line_protocol, validate_canonical_tags,
)
from notebooks._common.connection import load_env, get_influx_client, get_default_bucket
from notebooks._common.plotting import setup_default_style, plot_timeseries, plot_distribution
import notebooks._common.synthetic_mocks as mocks

SEED = 42
rng = np.random.default_rng(SEED)
setup_default_style()
load_env()
print(f"ROOT={ROOT}, SEED={SEED}, default_bucket={get_default_bucket()}")


## 9. Carga de datos o mock

Cargamos.


In [ ]:
csv_path = ROOT / "notebooks/_data/traffic_camera_mock.csv"
df = pd.read_csv(csv_path, comment="#", parse_dates=["timestamp"])
df.head()


## 10. Exploración paso a paso

Resumen por cámara.


In [ ]:
print(df.groupby("camera_id")["vehicle_count"].agg(["mean", "max"]).round(2))


## 11. Transformación bronce → plata

Generamos line protocol.


In [ ]:
out_dir = ROOT / "output" / "case_J"
out_dir.mkdir(parents=True, exist_ok=True)
lines = []
for _, row in df.iterrows():
    ts_ns = int(pd.Timestamp(row["timestamp"]).value)
    for csv_col, var in [("vehicle_count", "vehicle_count"),
                          ("congestion_level", "congestion_level"),
                          ("detection_confidence", "detection_confidence")]:
        lines.append(build_line_protocol(
            measurement=MEASUREMENT_TELEMETRY,
            tags={"captia_env": "dev", "domain_id": "traffic_cameras",
                  "site_id": "valencia", "asset_id": row["camera_id"], "variable": var},
            fields={"value": float(row[csv_col])},
            timestamp_ns=ts_ns,
        ))
(out_dir / "traffic.lp").write_text("\n".join(lines), encoding="utf-8")
print(f"Wrote {len(lines)} líneas")


## 12. Construcción de capa oro

Notebook 04.


## 13. Visualizaciones explicativas

Conteo medio por hora.


In [ ]:
df["hour"] = df["timestamp"].dt.hour
df.groupby("hour")["vehicle_count"].mean().plot.bar(color="#3F51B5", figsize=(7, 3))
plt.title("Tráfico medio por hora — todas las cámaras"); plt.tight_layout()


## 14. Validaciones

Tags presentes y valor numérico.


In [ ]:
sample = lines[0]
assert "domain_id=traffic_cameras" in sample
assert "value=" in sample
print("Sample:", sample)


## 15. Errores comunes

1. Mezclar imágenes con line protocol.
2. Usar `vehicle_count` como bool (>50 → True): pierde granularidad.
3. Olvidar `congestion_level` como variable independiente.


## 16. Ejercicios propuestos

1. Calcula `dvc/dt` en 15 min.
2. Define una alerta `congestion_level == 3` durante 30 min.
3. Combina con AEMET (lluvia) — ya en `traffic_camera_mock.csv`.


## 17. Cómo se reutiliza con datos reales

Sustituir el mock por counts del notebook 02.


## 18. Resumen final y próximos pasos

Recuerda los conceptos principales del notebook y enlaza al siguiente paso.

- Siguiente notebook: `10_case_J_traffic_yolo/04_integracion_meteo_trafico.ipynb`.
- Documento web del caso: `docs/use-cases/case-j-traffic-yolo.md`.
